# Offline Mode (Self-Managed vLLM) — Speculative Decoding Training

This notebook demonstrates how to train a custom **Eagle3 draft model** for speculative
decoding using the `OFFLINE` mode of `SpeculativeDecodingTrainer` from the Kubeflow SDK
on Red Hat OpenShift AI.

## What is OFFLINE Mode?

The `OFFLINE` mode connects to a **self-managed external vLLM server** to extract hidden
states from the verifier model, then trains the Eagle3 draft model — all within a single
job. Unlike `ONLINE` mode, the SDK does **not** deploy a vLLM sidecar. You provide a
`vllm_endpoint` pointing to your own vLLM instance.

This is useful when you already have a vLLM deployment running (e.g., as an OpenShift AI
model serving instance) and want to reuse it for hidden state extraction.

## How It Works

1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC
3. Training runs immediately after extraction completes — all within the same job

## Speculative Decoding Overview

Large language models generate tokens one at a time, and each token requires reading the
entire model from GPU memory — making inference **memory-bound**. Speculative decoding
exploits this: a small, fast **draft model** (~1.2 GB with Qwen3-0.6B) guesses the next several tokens,
then the large **verifier model** checks all guesses in a single forward pass. The output
is mathematically identical to normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate
layers of the verifier (not just the final logits), giving it richer context for more
accurate predictions.

## Dataset

This example uses the `magpie` built-in dataset (Magpie-format conversation dataset).

## Hardware Requirements

The table below shows the **minimum** resources needed. See the Configuration cell for
recommended values that improve training speed.

| Component | GPU (min) | GPU (recommended) | CPU (min) | CPU (rec.) | Memory (min) | Memory (rec.) |
|-----------|-----------|-------------------|-----------|------------|-------------|--------------|
| Training container | 1× GPU | 2× GPU | 1 core | 4 cores | 32Gi | 64Gi |
| External vLLM server | 1× GPU | 1× GPU | 1 core | 4 cores | 48Gi | 96Gi |

> The external vLLM server is self-managed — its resources are separate from the TrainJob.

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth
from kubernetes import client as k8s

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

The following environment variables are required for API authentication:

- `OPENSHIFT_API_URL` — your cluster API URL (e.g., `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` — an access token for API calls

In OpenShift AI workbenches, these are often auto-set.

If they are not set in your environment, uncomment and populate the values in the next cell.

In [ ]:
# ============================================================================
# AUTHENTICATION
# ============================================================================
# If your workbench does not auto-populate these env vars, uncomment and fill them in:
#
# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set. "
        "Either set them in your environment or uncomment the values above."
    )

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# Configure Kubernetes client
configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False  # Set to True if using trusted certificates
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# PVC MOUNT PATHS
# ============================================================================
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        "Expected workbench PVC mount not found at: "
        f"{NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name/mount, update PVC_NAME/NOTEBOOK_SHARED_PATH.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT AND CLUSTER TRAINING RUNTIME
# ============================================================================
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(client_configuration=configuration)
)

# ClusterTrainingRuntime (CTR) for OFFLINE mode.
# OFFLINE mode does not use a managed vLLM sidecar, so only the model optimization CTR is needed.
MODEL_OPT_CTR = "speculator-model-opt-cuda"  # Training only, no vLLM sidecar

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found" if MODEL_OPT_CTR in available_runtimes else "WARNING: not found on cluster"
)
print(f"CTR '{MODEL_OPT_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## (Optional) Download the Verifier Model

OFFLINE mode requires the verifier model on the shared PVC (as a PVC URI). If the
model is not already on your PVC, download it here. The external vLLM server must
also have access to this same model on the shared PVC.

Skip this cell if the model is already on your PVC.

In [ ]:
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = HF_TOKEN

model_id = "Qwen/Qwen3-0.6B"
local_dir = f"{NOTEBOOK_SHARED_PATH}/models/Qwen3-0.6B"

snapshot_download(model_id, local_dir=local_dir)
print(f"Model downloaded to {local_dir}")

## Configuration

The following constants configure the training run. The verifier model is
[Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), a 28-layer transformer,
specified as a PVC URI since OFFLINE mode requires the model on shared storage.

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

You must also set `VLLM_ENDPOINT` to point to your external vLLM server serving
the same verifier model.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
# OFFLINE mode with an external vLLM endpoint requires a PVC URI — the external
# vLLM server already has the model loaded from shared storage, so the training
# pod reads the model config from the same PVC path.
VERIFIER_MODEL_PVC_URI = f"pvc://{PVC_NAME}/models/Qwen3-0.6B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-0.6B has 28 transformer layers (indexed 1-28).
# Layers chosen: early (2), mid (14), late (25), and final (28) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
# When using a PVC URI for verifier_model, target_layer_ids MUST be set explicitly
# because the SDK cannot auto-detect them from the PVC.
TARGET_LAYER_IDS = [2, 14, 25, 28]

# Minimum resources for the training container.
# 1 GPU is sufficient to train the small Eagle3 draft model (~1.2 GB with Qwen3-0.6B).
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 1,  # Recommended: 2 — enables data-parallel training
    "cpu": "1",  # Recommended: "4" — faster data loading and preprocessing
    "memory": "32Gi",  # Recommended: "64Gi" — more headroom for optimizer state
}

# URL of your externally managed vLLM server.
# This must be a running vLLM instance serving the same verifier model (Qwen3-0.6B).
# The /v1 path exposes the OpenAI-compatible API that the SDK calls for extraction.
VLLM_ENDPOINT = "http://vllm-svc.speculative-decoding.svc.cluster.local:8000/v1"

# Training hyperparameters
EPOCHS = 3  # Number of full passes over the training data
LEARNING_RATE = 1e-4  # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for both extraction and training
MAX_SAMPLES = 500  # Cap on the number of dataset samples to process

print("OFFLINE Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL_PVC_URI}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  vLLM endpoint:     {VLLM_ENDPOINT}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## Offline Mode (Self-Managed vLLM)

The `OFFLINE` mode extracts hidden states via a self-managed external vLLM server,
then trains the draft model in a single job. This is useful when you already have a
vLLM deployment running (e.g., as an OpenShift AI model serving instance) and want to
reuse it for hidden state extraction instead of having the SDK deploy a sidecar.

**How it works:**
1. The job connects to your external vLLM endpoint to extract hidden states
2. Hidden states are saved to the PVC at `hidden_states_path`
3. Training runs immediately after extraction completes — all within the same job

**Key differences from other modes:**
- You must provide `vllm_endpoint` pointing to your external vLLM server
- The SDK does not deploy a vLLM sidecar — `vllm_resources` is not used
- Both `training_resources` (for the training container) and `vllm_endpoint`
  (for extraction) are required
- The external vLLM server must be in the **same namespace** and have access to the
  **same shared PVC** as the TrainJob

We use the `magpie` built-in dataset for this example.

In [ ]:
OFFLINE_JOB = f"eagle3-offline-{RUN_NAME}"
OFFLINE_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}-offline"

# Configure the OFFLINE trainer.
# OFFLINE mode connects to an external vLLM endpoint for extraction, then trains.
# Both steps happen within the same job — extraction first, training second.
# Unlike DATA_ONLY + TRAIN_ONLY, this is a single-job workflow.
# Unlike ONLINE, the SDK does NOT deploy a vLLM sidecar — you manage it yourself.
offline_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.OFFLINE,
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL_PVC_URI,
    dataset_name="magpie",  # Built-in Magpie conversation dataset
    max_samples=MAX_SAMPLES,
    total_seq_len=TOTAL_SEQ_LEN,
    vllm_endpoint=VLLM_ENDPOINT,  # External vLLM server URL
    hidden_states_path=f"{OFFLINE_OUTPUT}/hidden_states",  # Where extracted states are saved
    training_resources=TRAINING_RESOURCES,  # Resources for the training container
    regenerate_responses=True,  # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    output_dir=OFFLINE_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        resume_from_checkpoint=True,  # Resume from the latest checkpoint if one exists
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("OFFLINE Configuration:")
print(f"  Job name:         {OFFLINE_JOB}")
print(f"  Mode:             {offline_trainer.mode.value}")
print(f"  Verifier:         {offline_trainer.verifier_model}")
print(f"  vLLM endpoint:    {offline_trainer.vllm_endpoint}")
print(f"  Dataset:          {offline_trainer.dataset_name}")
print(f"  Target layers:    {offline_trainer.config.target_layer_ids}")
print(f"  Hidden states:    {offline_trainer.hidden_states_path}")
print(f"  Output dir:       {offline_trainer.output_dir}")

In [ ]:
# Submit the OFFLINE TrainJob to the cluster.
# Uses MODEL_OPT_CTR — no SDK-managed vLLM sidecar (the external endpoint handles extraction).
trainer_client.train(
    options=[Name(name=OFFLINE_JOB)],
    trainer=offline_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"OFFLINE job submitted: {OFFLINE_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={OFFLINE_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the OFFLINE job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(OFFLINE_JOB)

## Validate Trained Draft Model

Once training completes, load the latest checkpoint and run sample inference to validate the trained draft model performs as expected.

In [ ]:
import json

# Verify training checkpoint exists and contains model files
checkpoint_dir = f"{NOTEBOOK_SHARED_PATH}/speculator/{RUN_NAME}-offline"
checkpoint_best = os.path.join(checkpoint_dir, "checkpoint_best")

if os.path.islink(checkpoint_best):
    target = os.readlink(checkpoint_best)
    actual_checkpoint = os.path.join(checkpoint_dir, target)
    print(f"✓ checkpoint_best -> {target}")
else:
    print("⚠ checkpoint_best symlink not found")
    actual_checkpoint = None

if actual_checkpoint and os.path.exists(actual_checkpoint):
    # Verify required model files exist
    required_files = ["config.json", "model.safetensors", "generation_config.json"]
    files_found = [
        f for f in required_files if os.path.exists(os.path.join(actual_checkpoint, f))
    ]

    print(f"✓ Checkpoint directory: {actual_checkpoint}")
    print(f"✓ Model files found: {files_found}")

    # Model file size
    model_path = os.path.join(actual_checkpoint, "model.safetensors")
    if os.path.exists(model_path):
        size_gb = os.path.getsize(model_path) / (1024**3)
        print(f"✓ Model size: {size_gb:.2f} GB")

    # Load and display config
    config_path = os.path.join(actual_checkpoint, "config.json")
    if os.path.exists(config_path):
        with open(config_path) as f:
            config = json.load(f)
        print(f"✓ Model architecture: {config.get('architectures', ['unknown'])[0]}")
        print(f"✓ Speculator version: {config.get('speculators_version')}")

        # Eagle3-specific config
        if "eagle_aux_hidden_state_layer_ids" in config:
            print(
                f"  - Aux layer IDs: {config.get('eagle_aux_hidden_state_layer_ids')}"
            )
        if "speculative_tokens" in config:
            print(f"  - Speculative tokens: {config.get('speculative_tokens')}")
        if "draft_vocab_size" in config:
            print(f"  - Draft vocab size: {config.get('draft_vocab_size')}")

    # Check generation config
    gen_config_path = os.path.join(actual_checkpoint, "generation_config.json")
    if os.path.exists(gen_config_path):
        with open(gen_config_path) as f:
            gen_config = json.load(f)
        print(
            f"✓ Generation: max_length={gen_config.get('max_length')}, temperature={gen_config.get('temperature')}"
        )

    # Load validation metrics
    metrics_path = os.path.join(actual_checkpoint, "val_metrics.json")
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            metrics = json.load(f)
        # Get final epoch metrics (last entries)
        loss = (
            metrics.get("loss_2_epoch")
            or metrics.get("loss_1_epoch")
            or metrics.get("loss_0_epoch")
        )
        acc_full = (
            metrics.get("full_acc_2_epoch")
            or metrics.get("full_acc_1_epoch")
            or metrics.get("full_acc_0_epoch")
        )
        acc_cond = (
            metrics.get("cond_acc_2_epoch")
            or metrics.get("cond_acc_1_epoch")
            or metrics.get("cond_acc_0_epoch")
        )
        print("✓ Validation metrics:")
        if loss is not None:
            print(f"  - Loss: {loss:.4f}")
        if acc_full is not None:
            print(f"  - Full accuracy: {acc_full:.4f}")
        if acc_cond is not None:
            print(f"  - Conditional accuracy: {acc_cond:.4f}")

    print("\n✓ Training validation successful — checkpoint ready for serving")
else:
    print(f"⚠ Checkpoint not found at {checkpoint_best}")
    print(
        f"Verify training completed successfully with: oc logs -f -l batch.kubernetes.io/job-name={OFFLINE_JOB}-node-0 -c node"
    )

## (Optional) Serve the Trained Draft Model

Deploy the trained draft model using vLLM as an inference service on OpenShift AI.
This exposes a API endpoint for speculative decoding inference.

In [ ]:
# If your workbench does not auto-populate this env variable, uncomment and fill it in:
# NAMESPACE = "your-namespace"

NAMESPACE = os.getenv("NAMESPACE", "default")

# Note: You must have permission to create InferenceServices in this namespace.
# If you get a 403 error, contact your cluster admin or use a namespace where you have appropriate permissions.

# Define the KServe InferenceService to serve the trained draft model with vLLM
inference_service_yaml = """
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: eagle3-draft-{RUN_NAME}
  namespace: {NAMESPACE}
spec:
  predictor:
    model:
      modelFormat:
        name: vllm
      storageUri: {OFFLINE_OUTPUT}
      runtime: kserve-vllm-runtime
      resources:
        limits:
          nvidia.com/gpu: 1
        requests:
          memory: "16Gi"
          cpu: "4"
  transforMer:
    disabled: true
"""

print("InferenceService configuration:")
print(inference_service_yaml)
print("\nTo deploy the service, apply this YAML:")
print("  oc apply -f - <<EOF")
print(inference_service_yaml)
print("EOF")


## (Optional) Deploy as KServe InferenceService

Deploy the trained draft model as a KServe InferenceService for vLLM inference with speculative decoding.

In [ ]:
import json
import time

from kubernetes import dynamic
from kubernetes.client import api_client as k8s_api_client

# Deploy the trained draft model as a KServe InferenceService
service_name = f"eagle3-draft-{RUN_NAME}"

# If your workbench does not auto-populate this env variable, uncomment and fill it in:
# NAMESPACE = "your-namespace"

NAMESPACE = os.getenv("NAMESPACE", "default")

# Create InferenceService manifest
inference_service = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": service_name,
        "namespace": NAMESPACE,
    },
    "spec": {
        "predictor": {
            "model": {
                "modelFormat": {
                    "name": "vllm",
                },
                "storageUri": f"{OFFLINE_OUTPUT}/checkpoint_best",
                "runtime": "kserve-vllm-runtime",
                "resources": {
                    "limits": {
                        "nvidia.com/gpu": 1,
                    },
                    "requests": {
                        "memory": "16Gi",
                        "cpu": "4",
                    },
                },
            },
        },
    },
}

# Apply the InferenceService using dynamic k8s client
try:
    dynamic_client = dynamic.DynamicClient(
        k8s_api_client.ApiClient(configuration=configuration)
    )
    api = dynamic_client.resources.get(
        group="serving.kserve.io",
        api_version="v1beta1",
        kind="InferenceService",
    )

    # Create or replace the InferenceService
    result = api.create(body=inference_service, namespace=NAMESPACE)
    print("✓ InferenceService '{service_name}' created")

    # Poll for readiness (timeout after 10 minutes)
    print("Waiting for InferenceService to become ready...")
    start_time = time.time()
    timeout = 600  # 10 minutes
    poll_interval = 10  # 10 seconds

    while time.time() - start_time < timeout:
        try:
            is_status = api.get(name=service_name, namespace=NAMESPACE)
            conditions = is_status.get("status", {}).get("conditions", [])
            ready_condition = next(
                (c for c in conditions if c.get("type") == "Ready"), None
            )

            if ready_condition and ready_condition.get("status") == "True":
                service_url = is_status["status"].get("url")
                print("✓ InferenceService ready at: {service_url}")
                break
        except Exception:
            pass

        time.sleep(poll_interval)
    else:
        print("⚠ InferenceService did not become ready within {timeout} seconds")
        print(
            "Check status with: oc get inferenceservice {service_name} -n {NAMESPACE}"
        )

except Exception as _:
    print("⚠ Error deploying InferenceService: {e}")
    print("Deploy manually with: oc apply -f - <<EOF")
    print(json.dumps(inference_service, indent=2))
    print("EOF")

## Cleanup

Delete the TrainJob when you are done.


In [ ]:
# Delete the completed TrainJob and inference service to free cluster resources.
# Note: Deleting these does NOT delete the output data on the PVC —
# checkpoints remain available for future use.

# Delete training job
# trainer_client.delete_job(OFFLINE_JOB)
# print(f"TrainJob '{OFFLINE_JOB}' deleted")

# Delete inference service
# SERVICE_NAME = f"eagle3-draft-{RUN_NAME}"

# If your workbench does not auto-populate this env variable, uncomment and fill it in:
# NAMESPACE = "your-namespace"
# NAMESPACE = os.getenv("NAMESPACE", "default")

# Note: You must have permission to create InferenceServices in this namespace.
# If you get a 403 error, contact your cluster admin or use a namespace where you have appropriate permissions.

# for kind in ["inferenceservice", "servingruntime"]:
#     result = subprocess.run(
#         ["oc", "delete", kind, SERVICE_NAME, "-n", NAMESPACE, "--ignore-not-found"],
#         capture_output=True,
#         text=True,
#     )
#     if result.returncode == 0:
#         print(f"  Deleted {kind} '{SERVICE_NAME}'")

## Summary

This notebook demonstrated **OFFLINE** mode — extracting hidden states via an external
vLLM endpoint and training an Eagle3 draft model in a single job using the `magpie`
dataset.

OFFLINE mode is ideal when you already have a vLLM deployment running and want to
reuse it for hidden state extraction instead of having the SDK deploy a managed sidecar.

### Key Takeaways

- `vllm_endpoint` points to your external vLLM server — the SDK does not deploy a sidecar
- Both extraction and training happen in a single job
- The external vLLM server must be serving the same verifier model used in training
- All storage paths use **PVC URIs** (`pvc://<pvc-name>/<path>`)

### Next Steps

- Deploy the trained draft model with vLLM for speculative decoding inference
- Adjust `epochs`, `lr`, and `max_samples` to tune draft model quality
- Try [DATA_ONLY + TRAIN_ONLY](../data-only/) to separate extraction from training
  and iterate on hyperparameters without re-running extraction
- Try [ONLINE](../online/) mode for the fully managed alternative where the SDK
  deploys the vLLM sidecar automatically